# Case Study 1: E-Commerce Sales Analysis (Python/Pandas Edition)

**Data:** Kaggle – Online Retail Dataset
**File:** OnlineRetail.csv
**Columns:** InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

**Goal:** Use Python (pandas, scipy) to analyze customer purchasing behavior, uncover spending patterns, and apply basic inferential statistics (like t-tests) to make business insights.

task 1

Open OnlineRetail.csv and filter to display only the transactions made by customers from 'United Kingdom'. Count how many unique CustomerIDs made purchases from the UK.

In [1]:
import pandas as pd

df = pd.read_csv("OnlineRetail.csv")

print(f"Total transactions in dataset: {len(df)}")
print(f"Total unique invoices: {df['InvoiceNo'].nunique()}")
print(df.head())

# Filter to only United Kingdom transactions (equivalent to Excel's Filter feature)
uk_transactions = df[df["Country"] == "United Kingdom"]

print(f"\nNumber of UK transactions: {len(uk_transactions)}")

# Count unique CustomerIDs from the UK
uk_unique_customers = uk_transactions["CustomerID"].nunique()

print(f"Number of unique UK CustomerIDs: {uk_unique_customers}")


Total transactions in dataset: 847
Total unique invoices: 278
   InvoiceNo StockCode                         Description  Quantity  \
0     536365     71053                 WHITE METAL LANTERN        36   
1     536365     47566                       PARTY BUNTING        22   
2     536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER        22   
3     536366     22632           HAND WARMER RED POLKA DOT         1   
4     536366     22752        SET 7 BABUSHKA NESTING BOXES        28   

        InvoiceDate  UnitPrice CustomerID         Country  
0  02/17/2011 14:00       3.39     UK1017  United Kingdom  
1  02/17/2011 14:00       4.95     UK1017  United Kingdom  
2  02/17/2011 14:00       2.55     UK1017  United Kingdom  
3  03/26/2011 10:00       1.85     UK1003  United Kingdom  
4  03/26/2011 10:00       7.65     UK1003  United Kingdom  

Number of UK transactions: 375
Number of unique UK CustomerIDs: 37


**Explanation:** `df[df["Country"] == "United Kingdom"]` is the pandas equivalent of using Excel's Filter feature on the Country column. `.nunique()` on the CustomerID column then counts how many *distinct* customers appear in that filtered UK data, giving the same result as manually filtering and counting unique values in Excel.

task 2

Create a PivotTable-style summary to analyze total sales (Quantity * UnitPrice) by Country. Identify the top 3 countries by total sales.

In [2]:
# Add a LineTotal column = Quantity * UnitPrice (this is what would go into the PivotTable's Values area)
df["LineTotal"] = df["Quantity"] * df["UnitPrice"]

# Equivalent of an Excel PivotTable: Country in Rows, Sum of LineTotal in Values
sales_by_country = df.groupby("Country")["LineTotal"].sum().sort_values(ascending=False)

print("Total sales by country:")
print(sales_by_country)

top_3_countries = sales_by_country.head(3)
print("\nTop 3 countries by total sales:")
for rank, (country, total) in enumerate(top_3_countries.items(), start=1):
    print(f"  #{rank}: {country} - {round(total, 2)}")


Total sales by country:
Country
United Kingdom    32770.00
Germany           27773.22
Netherlands       12739.34
France            11646.93
Spain              6447.23
Belgium            4815.25
EIRE               3568.48
Name: LineTotal, dtype: float64

Top 3 countries by total sales:
  #1: United Kingdom - 32770.0
  #2: Germany - 27773.22
  #3: Netherlands - 12739.34


**Explanation:** `df.groupby("Country")["LineTotal"].sum()` does exactly what an Excel PivotTable would do with Country dragged into Rows and LineTotal (Quantity x UnitPrice) dragged into Values as Sum — it aggregates total sales for each country. Sorting this in descending order and taking the top 3 rows gives the highest-selling countries directly, without needing to manually scan a PivotTable.

task 3

Calculate the average order value (total sales per InvoiceNo) for all customers. Highlight any InvoiceNo where the order value is above ₹10,000.

In [3]:
# Order value per invoice = sum of LineTotal for all line items belonging to that invoice
order_value = df.groupby("InvoiceNo")["LineTotal"].sum().reset_index()
order_value.columns = ["InvoiceNo", "OrderValue"]

average_order_value = order_value["OrderValue"].mean()
print(f"Average order value across all invoices: {round(average_order_value, 2)}")

# "Highlight" (in a notebook, we flag/filter) invoices with order value above 10,000
high_value_orders = order_value[order_value["OrderValue"] > 10000]

print(f"\nNumber of invoices with order value above 10,000: {len(high_value_orders)}")
print("\nHigh-value invoices (order value > 10,000):")
print(high_value_orders)

# Style the DataFrame to visually highlight these rows (renders as a colored table in Jupyter)
def highlight_high_value(row):
    color = "background-color: #FFC7CE" if row["OrderValue"] > 10000 else ""
    return [color] * len(row)

styled_order_value = order_value.sort_values("OrderValue", ascending=False).head(15).style.apply(highlight_high_value, axis=1)
styled_order_value


Average order value across all invoices: 358.85

Number of invoices with order value above 10,000: 1

High-value invoices (order value > 10,000):
     InvoiceNo  OrderValue
160     536525    10493.02


,InvoiceNo,OrderValue
160,536525,10493.020000
211,536576,8627.600000
192,536557,5243.250000
269,536634,3213.000000
57,536422,2974.950000
121,536486,2393.750000
16,536381,1868.500000
129,536494,973.500000
17,536382,685.900000
86,536451,633.200000


**Explanation:** Grouping by `InvoiceNo` and summing `LineTotal` gives the total order value for each invoice — the pandas equivalent of creating a new "Order Value" column in Excel and using SUMIF. Instead of Excel's Conditional Formatting, we filter directly for orders above 10,000 with a boolean condition, and additionally use pandas' `.style.apply()` to visually color-highlight the high-value rows directly in the notebook output, mirroring what conditional formatting would look like in a spreadsheet.

task 4

Perform a t-test to compare the average order value between customers from 'France' and 'Germany'. State whether the difference is statistically significant at the 0.05 level.

In [4]:
from scipy import stats

# Need to know which country each invoice belongs to (each invoice has one country)
invoice_country = df.drop_duplicates(subset="InvoiceNo")[["InvoiceNo", "Country"]]
order_value_with_country = order_value.merge(invoice_country, on="InvoiceNo")

france_orders = order_value_with_country[order_value_with_country["Country"] == "France"]["OrderValue"]
germany_orders = order_value_with_country[order_value_with_country["Country"] == "Germany"]["OrderValue"]

print(f"France - number of orders: {len(france_orders)}, mean order value: {round(france_orders.mean(), 2)}")
print(f"Germany - number of orders: {len(germany_orders)}, mean order value: {round(germany_orders.mean(), 2)}")

# Two-sample t-test assuming unequal variances (Welch's t-test) - same as Excel's
# Data Analysis Toolpak "t-Test: Two-Sample Assuming Unequal Variances"
t_stat, p_value = stats.ttest_ind(france_orders, germany_orders, equal_var=False)

print(f"\nt-statistic = {round(t_stat, 4)}")
print(f"p-value = {round(p_value, 4)}")

alpha = 0.05
if p_value < alpha:
    conclusion = "Reject H0 -> the difference IS statistically significant at the 0.05 level"
else:
    conclusion = "Fail to reject H0 -> the difference is NOT statistically significant at the 0.05 level"

print(f"Conclusion: {conclusion}")


France - number of orders: 36, mean order value: 323.53
Germany - number of orders: 48, mean order value: 578.61

t-statistic = -1.0366
p-value = 0.3045
Conclusion: Fail to reject H0 -> the difference is NOT statistically significant at the 0.05 level


**Explanation:** `scipy.stats.ttest_ind(..., equal_var=False)` performs exactly the same test as Excel's Data Analysis Toolpak "t-Test: Two-Sample Assuming Unequal Variances" (also known as Welch's t-test) — comparing the mean order value of France vs Germany without assuming their variances are equal. Since the resulting p-value is above 0.05, we fail to reject the null hypothesis: there isn't strong statistical evidence that average order value differs between French and German customers, even though Germany's total sales (from Task 2) were higher — that's more likely explained by Germany having more orders overall, not larger orders on average.

task 5

Use ChatGPT or Copilot to generate a summary of key customer purchasing patterns you observe from your analysis (e.g., high-value countries, average order size, seasonal trends). Copy-paste your summary here.

In [5]:
ai_generated_summary = """
1. High-value countries: The United Kingdom is by far the largest market by total sales, which
   makes sense given it has the most customers and transactions in this dataset. Germany and the
   Netherlands are the next strongest markets, together forming a clear "second tier" behind the
   UK - suggesting these three countries should be the primary focus for retention and loyalty
   campaigns.

2. Average order size: The average order value across all customers is a useful benchmark for
   identifying unusually large or small transactions. A small number of invoices are dramatically
   larger than the rest (bulk/wholesale-style orders), which pull the average up above the typical
   (median) order - this right-skew is common in e-commerce data and means the median order value
   is often a more representative "typical" order size than the mean.

3. Country-level spending differences: Comparing average order values between France and Germany
   with a t-test did not show a statistically significant difference at the 0.05 level, meaning we
   don't have strong evidence that customers from one country spend more per order than the other -
   despite Germany's higher total sales, that is likely driven more by having more orders overall
   than by larger order sizes.

4. Seasonal / bulk order patterns: A few very large invoices (well above the typical order size)
   stand out sharply from the rest of the data. These are worth investigating individually - they
   may represent wholesale buyers, corporate gifting orders, or a special promotional period, and
   understanding what drives them could help the business identify and nurture similar high-value
   customers.

5. Recommendation: Since a small number of countries and a small number of large orders account
   for a disproportionate share of total revenue, the business could benefit from a tiered
   approach - dedicated account support for top countries (UK, Germany, Netherlands) and a
   separate outreach strategy to convert more customers into repeat, bulk-style buyers.
"""

my_analysis = (
    "My own analysis: The t-test result in Task 4 is a good reminder that a country with higher "
    "total sales (like Germany here) doesn't necessarily have customers who spend more per order - "
    "it may simply have more customers or more frequent orders, which is an important distinction "
    "for deciding whether a marketing campaign should focus on acquiring new customers versus "
    "increasing order value among existing ones."
)

print(ai_generated_summary)
print(my_analysis)



1. High-value countries: The United Kingdom is by far the largest market by total sales, which
   makes sense given it has the most customers and transactions in this dataset. Germany and the
   Netherlands are the next strongest markets, together forming a clear "second tier" behind the
   UK - suggesting these three countries should be the primary focus for retention and loyalty
   campaigns.

2. Average order size: The average order value across all customers is a useful benchmark for
   identifying unusually large or small transactions. A small number of invoices are dramatically
   larger than the rest (bulk/wholesale-style orders), which pull the average up above the typical
   (median) order - this right-skew is common in e-commerce data and means the median order value
   is often a more representative "typical" order size than the mean.

3. Country-level spending differences: Comparing average order values between France and Germany
   with a t-test did not show a statistical

**Explanation:** This task is conceptual, so the AI-generated summary and the added personal analysis are stored as text and printed. The summary ties together the concrete numbers computed in Tasks 1-4 (top countries, average order value, the France vs Germany t-test result) into a business-facing narrative that a non-technical stakeholder could act on.